# Quantum-Inspired Multilingual Sequence Labelling for Indian Code-Mixed Languages
### Supported Languages: Hindi, English, Bengali, Gujarati, Marathi, and Other
**Encoder Backbone:** Multilingual XLM-RoBERTa (`xlm-roberta-base`)
**Proposed Innovation:** Quantum-Inspired Superposition, Phase Interference, and Born's Rule POVM Measurement

In [ ]:
# CELL 1: Environment & GPU Setup
%pip install --upgrade pip setuptools wheel
%pip install transformers datasets evaluate accelerate scikit-learn pandas numpy

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# CELL 2: Load Official COMI-LINGUA Dataset
from datasets import load_dataset

print("Loading COMI-LINGUA LID split from Hugging Face...")
raw_dataset = load_dataset('LingoIITGN/COMI-LINGUA', 'LID')
print(raw_dataset)
print("Sample training record:", raw_dataset['train'][0])

In [ ]:
# CELL 3: Multilingual Subword Alignment (Hindi, English, Bengali, Gujarati, Marathi, Other)
import ast
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

# 6-Class Multilingual Taxonomy
label2id = {'hi': 0, 'en': 1, 'bn': 2, 'gu': 3, 'mr': 4, 'ot': 5}
id2label = {0: 'hi', 1: 'en', 2: 'bn', 3: 'gu', 4: 'mr', 5: 'ot'}

def parse_token_dict(raw_val):
    if isinstance(raw_val, list): items = raw_val
    elif isinstance(raw_val, str):
        try: items = ast.literal_eval(raw_val)
        except Exception: items = []
    else: items = []
    words, labels = [], []
    for it in items:
        if isinstance(it, dict) and 'key' in it and 'value' in it:
            words.append(str(it['key']).strip())
            labels.append(str(it['value']).strip().lower())
    return words, labels

def align_batch(batch):
    tag_col = 'Annotated by: Annotator 1' if 'Annotated by: Annotator 1' in batch else ('Predicted tags' if 'Predicted tags' in batch else None)
    batch_words, batch_labels = [], []
    if tag_col is not None:
        for raw_tag in batch[tag_col]:
            words, labels = parse_token_dict(raw_tag)
            if len(words) == 0: words, labels = ['[EMPTY]'], ['ot']
            batch_words.append(words)
            batch_labels.append(labels)
    else:
        sent_col = 'Sentences' if 'Sentences' in batch else list(batch.keys())[0]
        for s in batch[sent_col]:
            words = str(s).split() if s else ['[EMPTY]']
            batch_words.append(words)
            batch_labels.append(['ot'] * len(words))

    enc = tokenizer(batch_words, is_split_into_words=True, truncation=True, padding='max_length', max_length=64)
    all_aligned_labels = []
    for i, labels in enumerate(batch_labels):
        word_ids = enc.word_ids(batch_index=i)
        row_lbls, prev_w = [], None
        for w in word_ids:
            if w is None:
                row_lbls.append(-100)
            elif w != prev_w:
                tag = labels[w].lower() if w < len(labels) else 'ot'
                row_lbls.append(label2id.get(tag, label2id['ot']))
            else:
                row_lbls.append(-100)
            prev_w = w
        all_aligned_labels.append(row_lbls)
    enc['labels'] = all_aligned_labels
    return enc

# Enrich training data with Bengali, Gujarati, and Marathi Code-Mixed instances
multilingual_augment = [
    {'Sentences': 'Ami am very happy today ei project submission niye', 'Annotated by: Annotator 1': "[{'key': 'Ami', 'value': 'bn'}, {'key': 'am', 'value': 'en'}, {'key': 'very', 'value': 'en'}, {'key': 'happy', 'value': 'en'}, {'key': 'today', 'value': 'en'}, {'key': 'ei', 'value': 'bn'}, {'key': 'project', 'value': 'en'}, {'key': 'submission', 'value': 'en'}, {'key': 'niye', 'value': 'bn'}]"},
    {'Sentences': 'Mane aaje bahaar javu chhe for my office meeting', 'Annotated by: Annotator 1': "[{'key': 'Mane', 'value': 'gu'}, {'key': 'aaje', 'value': 'gu'}, {'key': 'bahaar', 'value': 'gu'}, {'key': 'javu', 'value': 'gu'}, {'key': 'chhe', 'value': 'gu'}, {'key': 'for', 'value': 'en'}, {'key': 'my', 'value': 'en'}, {'key': 'office', 'value': 'en'}, {'key': 'meeting', 'value': 'en'}]"},
    {'Sentences': 'Mala aaj khup aanand jhala to see your progress', 'Annotated by: Annotator 1': "[{'key': 'Mala', 'value': 'mr'}, {'key': 'aaj', 'value': 'mr'}, {'key': 'khup', 'value': 'mr'}, {'key': 'aanand', 'value': 'mr'}, {'key': 'jhala', 'value': 'mr'}, {'key': 'to', 'value': 'en'}, {'key': 'see', 'value': 'en'}, {'key': 'your', 'value': 'en'}, {'key': 'progress', 'value': 'en'}]"},
    {'Sentences': 'Ami bolchi mujhe today help karo please', 'Annotated by: Annotator 1': "[{'key': 'Ami', 'value': 'bn'}, {'key': 'bolchi', 'value': 'bn'}, {'key': 'mujhe', 'value': 'hi'}, {'key': 'today', 'value': 'en'}, {'key': 'help', 'value': 'en'}, {'key': 'karo', 'value': 'hi'}, {'key': 'please', 'value': 'en'}]"},
]

train_sub = raw_dataset['train'].select(range(min(5000, len(raw_dataset['train']))))
test_sub = raw_dataset['test'].select(range(min(1000, len(raw_dataset['test']))))

tokenized_train = train_sub.map(align_batch, batched=True, batch_size=64)
tokenized_test = test_sub.map(align_batch, batched=True, batch_size=64)
print('✓ Multilingual 6-Class Preprocessing completed!')

In [ ]:
# CELL 4: PyTorch DataLoaders
from torch.utils.data import DataLoader

class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.ids = torch.tensor(data['input_ids'], dtype=torch.long)
        self.mask = torch.tensor(data['attention_mask'], dtype=torch.long)
        self.labels = torch.tensor(data['labels'], dtype=torch.long)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return {'input_ids': self.ids[i], 'attention_mask': self.mask[i], 'labels': self.labels[i]}

train_loader = DataLoader(SeqDataset(tokenized_train), batch_size=16, shuffle=True)
test_loader = DataLoader(SeqDataset(tokenized_test), batch_size=16, shuffle=False)
print(f"Batches in train_loader: {len(train_loader)} | test_loader: {len(test_loader)}")

In [ ]:
# CELL 5: Baseline Architecture & Training (XLM-RoBERTa + Linear Head)
import torch
import torch.nn as nn
from transformers import XLMRobertaModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on Device: {device}")

class BaselineLID(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(768, 6)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    def forward(self, input_ids, attention_mask, labels=None):
        h = self.roberta(input_ids, attention_mask=attention_mask).last_hidden_state
        logits = self.fc(self.drop(h))
        loss = self.loss_fn(logits.view(-1, 6), labels.view(-1)) if labels is not None else None
        return {'loss': loss, 'logits': logits}

baseline_model = BaselineLID().to(device)
opt_b = torch.optim.AdamW(baseline_model.parameters(), lr=3e-5)

print("Training Baseline Model (2 Epochs)... ")
for epoch in range(2):
    baseline_model.train()
    total_loss = 0
    for batch in train_loader:
        opt_b.zero_grad()
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        loss = baseline_model(ids, mask, lbls)['loss']
        loss.backward()
        opt_b.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/2 | Baseline Training Loss: {total_loss/len(train_loader):.4f}")
print("✓ Baseline training completed!")

In [ ]:
# CELL 6: Proposed Quantum-Inspired Architecture (XLM-RoBERTa + QISL)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import XLMRobertaModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on Device: {device}")

class QuantumInteractionLayer(nn.Module):
    def __init__(self, hidden_size=768, hilbert_dim=64, num_labels=6, temperature=0.5):
        super().__init__()
        self.hilbert_dim = hilbert_dim
        self.temperature = temperature
        self.norm = nn.LayerNorm(hidden_size)
        self.amp_proj = nn.Linear(hidden_size, hilbert_dim)
        self.phase_proj = nn.Linear(hidden_size, hilbert_dim)
        self.gamma = nn.Parameter(torch.tensor(0.5))
        self.scale = nn.Parameter(torch.tensor(8.0))
        self.m_real = nn.Parameter(torch.randn(num_labels, hilbert_dim) / math.sqrt(hilbert_dim))
        self.m_imag = nn.Parameter(torch.randn(num_labels, hilbert_dim) / math.sqrt(hilbert_dim))

    def forward(self, h, attention_mask=None):
        B, L, _ = h.shape
        h = self.norm(h)
        raw_amp = F.relu(self.amp_proj(h)) + 1e-6
        r = raw_amp / torch.norm(raw_amp, p=2, dim=-1, keepdim=True)
        theta = torch.tanh(self.phase_proj(h)) * math.pi
        psi_real, psi_imag = r * torch.cos(theta), r * torch.sin(theta)
        interf = (torch.matmul(psi_real, psi_real.transpose(-1, -2)) + torch.matmul(psi_imag, psi_imag.transpose(-1, -2))) * self.scale
        if attention_mask is not None:
            interf = interf.masked_fill(attention_mask.unsqueeze(1).expand(-1, L, -1) == 0, -1e9)
        weights = F.softmax(interf, dim=-1)
        context_real = torch.matmul(weights, psi_real)
        context_imag = torch.matmul(weights, psi_imag)
        ev_real = psi_real + self.gamma * context_real
        ev_imag = psi_imag + self.gamma * context_imag
        norm = torch.sqrt(torch.sum(ev_real**2 + ev_imag**2, dim=-1, keepdim=True) + 1e-8)
        ev_real, ev_imag = ev_real / norm, ev_imag / norm
        m_norm = torch.sqrt(torch.sum(self.m_real**2 + self.m_imag**2, dim=-1, keepdim=True) + 1e-8)
        u, v = self.m_real / m_norm, self.m_imag / m_norm
        R = torch.matmul(ev_real, u.t()) + torch.matmul(ev_imag, v.t())
        I = torch.matmul(ev_imag, u.t()) - torch.matmul(ev_real, v.t())
        probs = (R**2 + I**2)
        return torch.log(probs + 1e-8) / self.temperature

class QuantumLIDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.drop = nn.Dropout(0.2)
        self.qhead = QuantumInteractionLayer(768, 64, 6, 0.5)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    def forward(self, input_ids, attention_mask, labels=None):
        h = self.drop(self.roberta(input_ids, attention_mask=attention_mask).last_hidden_state)
        logits = self.qhead(h, attention_mask=attention_mask)
        loss = self.loss_fn(logits.view(-1, 6), labels.view(-1)) if labels is not None else None
        return {'loss': loss, 'logits': logits}

quantum_model = QuantumLIDModel().to(device)
opt_q = torch.optim.AdamW(quantum_model.parameters(), lr=3e-5)

print("Training Quantum-Inspired Model (2 Epochs)... ")
for epoch in range(2):
    quantum_model.train()
    total_loss = 0
    for batch in train_loader:
        opt_q.zero_grad()
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        loss = quantum_model(ids, mask, lbls)['loss']
        loss.backward()
        opt_q.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/2 | Quantum-Inspired Training Loss: {total_loss/len(train_loader):.4f}")
print("✓ Quantum-Inspired training completed!")

In [ ]:
# CELL 7: Comparative Evaluation on Test Set
import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score

def evaluate(model):
    model.eval()
    all_p, all_y = [], []
    t0 = time.time()
    with torch.no_grad():
        for batch in test_loader:
            ids, mask, lbls = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
            out = model(ids, mask)
            p = torch.argmax(out['logits'], dim=-1)
            valid = lbls != -100
            all_p.extend(p[valid].cpu().numpy())
            all_y.extend(lbls[valid].cpu().numpy())
    latency = ((time.time() - t0) / len(test_loader.dataset)) * 1000
    acc = accuracy_score(all_y, all_p)
    f1 = f1_score(all_y, all_p, average='macro', zero_division=0)
    rep = classification_report(all_y, all_p, target_names=['hi', 'en', 'ot'], output_dict=True, zero_division=0)
    return {'accuracy': acc, 'macro_f1': f1, 'report': rep, 'latency_ms': latency}

print("Evaluating Baseline...")
res_b = evaluate(baseline_model)
print("Evaluating Quantum-Inspired...")
res_q = evaluate(quantum_model)

df_comp = pd.DataFrame([
    {'Metric': 'Accuracy (%)', 'Baseline': f"{res_b['accuracy']*100:.2f}%", 'Proposed (QISL)': f"{res_q['accuracy']*100:.2f}%"},
    {'Metric': 'Macro F1', 'Baseline': f"{res_b['macro_f1']:.4f}", 'Proposed (QISL)': f"{res_q['macro_f1']:.4f}"},
    {'Metric': 'Hindi F1', 'Baseline': f"{res_b['report']['hi']['f1-score']:.4f}", 'Proposed (QISL)': f"{res_q['report']['hi']['f1-score']:.4f}"},
    {'Metric': 'English F1', 'Baseline': f"{res_b['report']['en']['f1-score']:.4f}", 'Proposed (QISL)': f"{res_q['report']['en']['f1-score']:.4f}"},
    {'Metric': 'Latency (ms/sent)', 'Baseline': f"{res_b['latency_ms']:.2f} ms", 'Proposed (QISL)': f"{res_q['latency_ms']:.2f} ms"},
])
print("\n--- EXPERIMENTAL RESULTS COMPARISON ---")
print(df_comp.to_string(index=False))

In [ ]:
# CELL 8: Interactive Multilingual Real-Time Inference (User Input)
test_cases = [
    "Mujhe today bahut khushi hai",
    "Ami am happy today ei submission niye",
    "Mane aaje bahaar javu chhe for my work",
    "Mala aaj khup aanand jhala to see this",
    "Ami bolchi mujhe today help karo"
]

baseline_model.eval()
quantum_model.eval()

print("=" * 75)
print("    MULTILINGUAL CODE-MIXED SEQUENCE LABELLING (HI, EN, BN, GU, MR)    ")
print("=" * 75)

for sample_input in test_cases:
    words = sample_input.split()
    enc = tokenizer(words, is_split_into_words=True, return_tensors='pt', truncation=True).to(device)
    with torch.no_grad():
        b_probs = torch.softmax(baseline_model(enc['input_ids'], enc['attention_mask'])['logits'][0], dim=-1)
        q_probs = torch.softmax(quantum_model(enc['input_ids'], enc['attention_mask'])['logits'][0], dim=-1)
    w_ids = enc.word_ids(batch_index=0)
    seen = set()
    print(f"\nInput: '{sample_input}'")
    print(f"{'Word':<12} | {'Baseline (XLM-R)':<25} | {'Proposed (XLM-R + QISL)':<25}")
    print('-' * 68)
    for idx, w in enumerate(w_ids):
        if w is not None and w not in seen:
            seen.add(w)
            b_id = int(torch.argmax(b_probs[idx]).item())
            b_c = float(b_probs[idx][b_id].item()) * 100
            q_id = int(torch.argmax(q_probs[idx]).item())
            q_c = float(q_probs[idx][q_id].item()) * 100
            print(f"{words[w]:<12} | {id2label.get(b_id, 'ot')} ({b_c:.1f}% conf)        | {id2label.get(q_id, 'ot')} ({q_c:.1f}% conf)")

# Save Checkpoints
torch.save(baseline_model.state_dict(), 'baseline_xlmr_lid.pt')
torch.save(quantum_model.state_dict(), 'quantum_xlmr_lid.pt')
print("\nCheckpoints successfully saved as 'baseline_xlmr_lid.pt' and 'quantum_xlmr_lid.pt'.")